# Integrating neural networks in compartment based models

## Introduction

In this example, we will generate data from a 5 compartment model: Susceptible -> Infected -> Recovered -> Hospitalized -> Deaths. Following this, we will construct a Universal Differential Equation (UDE) by replacing each interaction term with a neural network. We will show the training and estimation using such a model. This example demonstrates the following things:

- How to write code which integrates neural networks with ODEs in Julia
- How to optimize the neural network parameters and train the resulting UDE
- Such a neural network assisted ODE framework has a wide range of applications, from quarantine diagnosis (https://www.sciencedirect.com/science/article/pii/S2666389920301938) to early reopening (https://spj.sciencemag.org/journals/hds/2021/9798302/) and virtual virus spread (https://www.sciencedirect.com/science/article/pii/S2666389921000349).

## Libraries

In [ ]:
Pollis.packages("Lux", "OrdinaryDiffEq", "SciMLSensitivity", "Optimization", "OptimizationOptimisers", "OptimizationOptimJL", "ComponentArrays", "Mooncake", "Plots")

In [ ]:
using Lux, OrdinaryDiffEq, SciMLSensitivity, Optimization, OptimizationOptimisers, OptimizationOptimJL, ComponentArrays, Mooncake, Random, Plots

## Data Generation
Generate data which simulates a Susceptible -> Infected -> Recovered -> Hospitalized -> Death scenario, as fractions of the population. Typically only the infected and the recovered are reported, so these two series, with a little noise, are the data the UDE is trained on.

In [ ]:
rng = Xoshiro(1)
N_days = 25

u0 = [0.99, 0.01, 0.0, 0.0, 0.0]

p0 = [
    0.85,   # tau_SI
    0.1,    # tau_IR
    0.025,  # tau_IH
    0.05,   # tau_ID
    0.02,   # tau_HR
    0.002,  # tau_HD
]

tspan = (0.0, Float64(N_days))
t = 0.0:1.0:N_days

function SIRHD!(du, u, p, t)
    S, I, R, H, D = u
    tau_SI, tau_IR, tau_IH, tau_ID, tau_HR, tau_HD = p
    du[1] = -tau_SI * S * I                                # S
    du[2] = tau_SI * S * I - (tau_IR + tau_IH + tau_ID) * I  # I
    du[3] = tau_IR * I + tau_HR * H                        # R
    du[4] = tau_IH * I - (tau_HR + tau_HD) * H             # H
    du[5] = tau_ID * I + tau_HD * H                        # D
end

prob = ODEProblem(SIRHD!, u0, tspan, p0)
sol = Array(solve(prob, Tsit5(); saveat = t))

Infected_Data = sol[2, :] .* (1 .+ 0.02 .* randn(rng, length(t)))
Recovered_Data = sol[3, :] .* (1 .+ 0.02 .* randn(rng, length(t)))

## Visualize the data

In [ ]:
plot(t, sol[1, :]; label = "Susceptible", xlabel = "Day", ylabel = "Fraction of the population")
plot!(t, sol[2, :]; label = "Infected")
plot!(t, sol[3, :]; label = "Recovered")
plot!(t, sol[4, :]; label = "Hospitalized")
plot!(t, sol[5, :]; label = "Deaths")
scatter!(t, Infected_Data; label = "Infected data", color = :red)
scatter!(t, Recovered_Data; label = "Recovered data", color = :blue)

## Constructing a Universal Differential Equation (UDE)
The simplest UDE we can construct replaces every interaction term in the SIRHD model with a neural network, while keeping the model's structure: what leaves one compartment enters another. We use Lux to define the six networks.

In [ ]:
NN1 = Chain(Dense(2 => 10, relu), Dense(10 => 1))  # S, I -> the flow from S to I
NN2 = Chain(Dense(1 => 10, relu), Dense(10 => 1))  # I -> I to R
NN3 = Chain(Dense(1 => 10, relu), Dense(10 => 1))  # I -> I to D
NN4 = Chain(Dense(1 => 10, relu), Dense(10 => 1))  # I -> I to H
NN5 = Chain(Dense(1 => 10, relu), Dense(10 => 1))  # H -> H to R
NN6 = Chain(Dense(1 => 10, relu), Dense(10 => 1))  # H -> H to D
networks = (NN1, NN2, NN3, NN4, NN5, NN6)
setups = [Lux.setup(rng, nn) for nn in networks]
states = Tuple(last.(setups))
p0_vec = ComponentArray(layer_1 = setups[1][1], layer_2 = setups[2][1], layer_3 = setups[3][1],
    layer_4 = setups[4][1], layer_5 = setups[5][1], layer_6 = setups[6][1])

flow(k, x, p) = abs(first(first(networks[k](x, p, states[k]))))

function dxdt_pred(u, p, t)
    S, I, R, H, D = u
    NNSI = flow(1, [S, I], p.layer_1)
    NNIR = flow(2, [I], p.layer_2)
    NNID = flow(3, [I], p.layer_3)
    NNIH = flow(4, [I], p.layer_4)
    NNHR = flow(5, [H], p.layer_5)
    NNHD = flow(6, [H], p.layer_6)
    return [
        -NNSI,                        # dS
        NNSI - NNIR - NNID - NNIH,    # dI
        NNIR + NNHR,                  # dR
        NNIH - NNHR - NNHD,           # dH
        NNID + NNHD,                  # dD
    ]
end

prob_pred = ODEProblem(dxdt_pred, u0, tspan)
predict_adjoint(theta, ts = t) = Array(solve(prob_pred, Tsit5(); p = theta, saveat = ts, tspan = (0.0, last(ts))))

# Only the infected and the recovered are observed, so the loss compares those two
function loss_adjoint(theta, _)
    x = predict_adjoint(theta)
    return sum(abs2, Infected_Data .- x[2, :]) + sum(abs2, Recovered_Data .- x[3, :])
end

losses = Float64[]
callback = function (state, l)
    push!(losses, l)
    length(losses) % 50 == 0 && println("Loss after ", length(losses), " iterations: ", l)
    return false
end

## Training
If the loss is stagnant or shoots to a very high value, the optimizer is stuck in a poor minimum: change the seed of `rng` so that the networks start from other weights, and run again.

In [ ]:
optf = OptimizationFunction(loss_adjoint, Optimization.AutoMooncake(; config = nothing))
optprob = OptimizationProblem(optf, p0_vec)
res1 = solve(optprob, Adam(0.05); callback = callback, maxiters = 300)
res2 = solve(remake(optprob; u0 = res1.u), Optim.BFGS(; initial_stepnorm = 0.01); callback = callback, maxiters = 200)
println("Final loss: ", losses[end])

plot(losses; yaxis = :log10, xlabel = "Iteration", ylabel = "Loss", label = "Training loss")

## Visualizing the predictions

In [ ]:
data_pred = predict_adjoint(res2.u)
plot(; legend = :topleft, xlabel = "Day", ylabel = "Fraction of the population")
bar!(t, Infected_Data; label = "I data", color = :red, alpha = 0.5)
bar!(t, Recovered_Data; label = "R data", color = :blue, alpha = 0.5)
plot!(t, data_pred[2, :]; label = "I prediction", color = :red, lw = 2)
plot!(t, data_pred[3, :]; label = "R prediction", color = :blue, lw = 2)

## The compartments that were not observed
The UDE also gives the hospitalized and the deaths, which it never saw. Compare them with the truth: the infected and the recovered alone say little about how the outflow of I splits between H and D, so expect these to be the least accurate.

In [ ]:
plot(t, sol[4, :] .+ sol[5, :]; label = "H + D (true)", ls = :dash, color = :black, xlabel = "Day", ylabel = "Fraction of the population")
plot!(t, data_pred[4, :] .+ data_pred[5, :]; label = "H + D (UDE)", lw = 2, color = :black)

## Discussion
The fitted model gives a good fit to the training data, and the training iterations proceed fast. The UDE keeps what is known, the flows between compartments, and learns only the rates; the next tutorial shows why this makes its forecasts much better than a Neural ODE's.